In [4]:
# %% [markdown]
# # Notebook 01: Load, Explore, Merge, Filter and Clean NHANES Data
#
# **Objective**:  
# 1. Load the NHANES data files (.xpt).  
# 2. **Explore columns of each file** to understand available variables.  
# 3. Merge only the essential core files (DEMO + RHQ) using `inner` join.  
# 4. Add other relevant modules (DPQ, SLQ, PAQ, ALQ, TST, DBQ, HSQ) using `left` join.  
# 5. Keep only female participants.  
# 6. Clean missing values.  
# 7. Save the cleaned dataset.
#
# **Author**: Sara YOUSSE
# **Date**: 2026-07-18

# %%
# Import necessary libraries
import pandas as pd
import numpy as np
import os

# Set pandas to show all columns when printing (so we see everything)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

# %%
# ------------------------------
# 1. Define file paths and names
# ------------------------------
# The notebook is inside the 'Model' folder. The data is in the sibling folder 'Data'.
data_path = "../Dataset/"          # relative path from the notebook to the Data folder
output_path = "../Dataset/"        # save cleaned data in the same Data folder

# List of all files we have (we will load them all for exploration)
all_files = [
    "DEMO_L.xpt",   # Demographics
    "RHQ_L.xpt",    # Reproductive Health
    "TST_L.xpt",    # Testosterone
    "ALQ_L.xpt",    # Alcohol
    "HSQ_L.xpt",    # Health Status
    "DBQ_L.xpt",    # Diabetes
    "DPQ_L.xpt",    # Depression (PHQ-9)
    "SLQ_L.xpt",    # Sleep
    "OCQ_L.xpt"     # Oral Health
]

# We will load all, but later we decide which to merge.
core_files = ["DEMO_L.xpt", "RHQ_L.xpt"]
optional_files = ["DPQ_L.xpt", "SLQ_L.xpt", "ALQ_L.xpt", "TST_L.xpt", "DBQ_L.xpt", "HSQ_L.xpt"]
ignored_files = ["OCQ_L.xpt"]  # Not relevant for menopause/psychology

# %%
# ------------------------------
# 2. Check that the Data folder exists
# ------------------------------
if not os.path.exists(data_path):
    os.makedirs(data_path)
    print(f"📁 Created folder: {data_path}")
else:
    print(f"✅ Data folder found at: {data_path}")

# %%
# ------------------------------
# 3. Load ALL files using pandas.read_sas
# ------------------------------
dfs = {}
for f in all_files:
    file_path = os.path.join(data_path, f)
    if not os.path.isfile(file_path):
        print(f"⚠️ File not found: {file_path} – skipping")
        continue
    try:
        df = pd.read_sas(file_path, format='xport')
        dfs[f] = df
        print(f"✅ Loaded: {f} -> {df.shape[0]} rows, {df.shape[1]} columns")
    except Exception as e:
        print(f"❌ Error loading {f}: {e}")

# %%
# ------------------------------
# 4. EXPLORE COLUMNS OF EACH FILE (before merging)
# ------------------------------
print("\n" + "="*80)
print("📋 COLUMN EXPLORATION FOR EACH FILE / EXPLORATION DES COLONNES")
print("="*80)

# A dictionary to store which columns belong to which file for later reference
column_summary = {}

for f, df in dfs.items():
    print(f"\n{'='*50}")
    print(f"📁 FILE: {f}")
    print(f"   Rows: {df.shape[0]}, Columns: {df.shape[1]}")
    print(f"   Column list:")
    # Print column names with index for easy reading
    for i, col in enumerate(df.columns):
        print(f"      {i+1:3d}. {col}")
    
    # Store the columns for later use
    column_summary[f] = df.columns.tolist()
    
    # Show a small sample for key files (to see what the data looks like)
    if f in ["DEMO_L.xpt", "RHQ_L.xpt", "DPQ_L.xpt"]:
        print(f"\n   📊 Sample (first 3 rows) for {f}:")
        print(df.head(3).to_string())

# %%
# ------------------------------
# 5. Identify key columns for our study
# ------------------------------
print("\n" + "="*80)
print("🔍 KEY VARIABLES IDENTIFIED / VARIABLES CLÉS IDENTIFIÉES")
print("="*80)

# From DEMO (demographics)
demo_cols = column_summary.get("DEMO_L.xpt", [])
print(f"\n👤 DEMO columns (age, income, education):")
demo_key = [c for c in demo_cols if any(key in c.upper() for key in ['AGE', 'SEX', 'GENDER', 'INDFMPIR', 'DMDEDUC'])]
print(f"   Potential candidates: {demo_key}")

# From RHQ (reproductive health - menopause)
rhq_cols = column_summary.get("RHQ_L.xpt", [])
print(f"\n👩 RHQ columns (reproductive / menopause):")
rhq_key = [c for c in rhq_cols if any(key in c.upper() for key in ['RHQ', 'RHD', 'MENOPAUSE', 'PREGNANT'])]
print(f"   Potential candidates: {rhq_key}")

# From DPQ (depression)
dpq_cols = column_summary.get("DPQ_L.xpt", [])
print(f"\n🧠 DPQ columns (depression / PHQ-9):")
dpq_key = [c for c in dpq_cols if 'DPQ' in c.upper()]
print(f"   Potential candidates: {dpq_key}")

# From SLQ (sleep)
slq_cols = column_summary.get("SLQ_L.xpt", [])
print(f"\n😴 SLQ columns (sleep):")
slq_key = [c for c in slq_cols if 'SLQ' in c.upper()]
print(f"   Potential candidates: {slq_key}")

# From TST (testosterone)
tst_cols = column_summary.get("TST_L.xpt", [])
print(f"\n🧬 TST columns (testosterone):")
tst_key = [c for c in tst_cols if any(key in c.upper() for key in ['TST', 'TESTO'])]
print(f"   Potential candidates: {tst_key}")

✅ Data folder found at: ../Dataset/
✅ Loaded: DEMO_L.xpt -> 11933 rows, 27 columns
✅ Loaded: RHQ_L.xpt -> 3917 rows, 13 columns
✅ Loaded: TST_L.xpt -> 8493 rows, 35 columns
✅ Loaded: ALQ_L.xpt -> 6337 rows, 9 columns
✅ Loaded: HSQ_L.xpt -> 6615 rows, 2 columns
✅ Loaded: DBQ_L.xpt -> 11933 rows, 27 columns
✅ Loaded: DPQ_L.xpt -> 6337 rows, 11 columns
✅ Loaded: SLQ_L.xpt -> 8501 rows, 7 columns
✅ Loaded: OCQ_L.xpt -> 8501 rows, 6 columns

📋 COLUMN EXPLORATION FOR EACH FILE / EXPLORATION DES COLONNES

📁 FILE: DEMO_L.xpt
   Rows: 11933, Columns: 27
   Column list:
        1. SEQN
        2. SDDSRVYR
        3. RIDSTATR
        4. RIAGENDR
        5. RIDAGEYR
        6. RIDAGEMN
        7. RIDRETH1
        8. RIDRETH3
        9. RIDEXMON
       10. RIDEXAGM
       11. DMQMILIZ
       12. DMDBORN4
       13. DMDYRUSR
       14. DMDEDUC2
       15. DMDMARTZ
       16. RIDEXPRG
       17. DMDHHSIZ
       18. DMDHRGND
       19. DMDHRAGZ
       20. DMDHREDZ
       21. DMDHRMAZ
       22. DMDHSE